**Notebook version: 2** — Claude will state the version number after editing any cell in this notebook.

# fiftyone_final_dataset.ipynb — browse the FINAL training dataset, all splits at once

Loads `dataset/final/{train,val,test}/` into a **single** FiftyOne dataset so the whole
36,875-image pool is browsable in one App session, with **`source` and `split` as real,
sidebar-filterable fields**.

This is the gap the other notebooks leave. `fiftyone_review_processed.ipynb` accepts
`source_key = "final/train"`, but it browses one split at a time and never sets a `source`
field — in `dataset/final/` the source survives only as the `<source>__` filename prefix
(`merge.py`/`prefixed_filename()`), which the App sidebar cannot filter on.

---

### This notebook is READ-ONLY, deliberately

There is no write-back cell here and there must not be one. `dataset/final/` is a **derived**
directory — `split.py` deletes and regenerates it wholesale on every cascade run, so any box
you fixed here would be silently destroyed the next time the cascade runs.

Corrections belong upstream, in `fiftyone_review_processed.ipynb`, which writes to
`dataset/processed/<source>/labels_reviewed/` and is promoted into `labels/` by
`promote_reviews.py` before the cascade. That is the only edit path that survives.

Use this notebook to **inspect and verify** — per-source class balance, what actually landed
in `val` (which Hailo's DFC uses for calibration), whether a source looks wrong at a glance.
Then go fix it upstream.


In [16]:
# Imports
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Same helper the other notebooks use -- notebooks live in notebooks/, and
    Jupyter's working directory depends on how it was launched.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import fiftyone as fo

from scripts.utils.config_loader import get_canonical_names
from scripts.utils.file_utils import final_dir, list_images, reports_dir

CANONICAL_NAMES = get_canonical_names()
print(f"repo root      : {REPO_ROOT}")
print(f"canonical nc   : {len(CANONICAL_NAMES)}")
print(f"canonical names: {CANONICAL_NAMES}")


repo root      : /Users/luna/Projects/Thesis/second-vision-ai
canonical nc   : 13
canonical names: ['Person', 'Vehicle', 'Motorcycle', 'Pole', 'Animals', 'Shelf', 'Doors', 'Chairs', 'Tables', 'Tricycle', 'Potholes', 'Trash Bins', 'Bicycle']


In [9]:
# ---- What to load ----

# Which splits to pull into the single browsable dataset. Keep all three for a
# whole-dataset view; narrow it if you only care about one (e.g. ["val"] when
# sanity-checking Hailo's calibration set specifically).
SPLITS = ["train", "val", "test"]

# Optional build-time narrowing. Leave both None to load everything and do all
# your filtering interactively in the App sidebar -- that is the intended
# workflow, and 36,875 images loads fine (labels + paths only, no image decoding).
#
# Set these only when you want a genuinely smaller dataset object, e.g. to hand
# one source to a colleague or to keep an App session snappy on a slow machine.
SOURCE_FILTER = None   # e.g. {"roboflow_pothole_voxrl", "dataset_ninja_pothole_detection"}
CLASS_FILTER = None    # e.g. {"Potholes"} -- keeps images containing >=1 box of these

# Cap per split, for a quick look without waiting on the full build. None = no cap.
MAX_IMAGES_PER_SPLIT = None

# Rebuilding drops and recreates the dataset. Safe here in a way it is NOT in
# fiftyone_review_processed.ipynb: this dataset holds no review work -- every
# field is re-derived from dataset/final/ on disk in about a minute.
DATASET_NAME = "final_dataset_browse"


In [10]:
# ---- Build one FiftyOne dataset spanning every requested split ----
#
# `source` and `split` are stored as top-level sample fields, which is the whole
# point of this notebook: both become sidebar filters in the App. The source is
# recovered from merge.py's filename prefix (prefixed_filename(): "<source>__<name>"),
# the same convention box_audit.py --pool merged already relies on (DEC-075).

def _source_of(filename: str) -> str:
    """Recover the source key from a merged/final filename prefix.

    merge.py prefixes every file it writes, so a name with no "__" means the
    pool was not built by merge.py -- surfaced rather than silently bucketed.
    """
    if "__" not in filename:
        return "<unprefixed>"
    return filename.split("__", 1)[0]


# Integrity check before trusting anything below: config/classes.yaml is the
# authoritative schema, dataset/final/data.yaml is what training actually reads.
# They are generated to agree (generate_yaml.py, DEC-065) -- if they have drifted,
# every count in this notebook is being labelled with the wrong names.
data_yaml_path = final_dir() / "data.yaml"
if data_yaml_path.is_file():
    import yaml
    _dy = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
    _dy_names = [_dy["names"][i] for i in sorted(_dy["names"])]
    if _dy_names != CANONICAL_NAMES:
        raise RuntimeError(
            f"dataset/final/data.yaml disagrees with config/classes.yaml.\n"
            f"  data.yaml    : {_dy_names}\n"
            f"  classes.yaml : {CANONICAL_NAMES}\n"
            f"Re-run scripts/build/generate_yaml.py before browsing."
        )
    print(f"data.yaml <-> classes.yaml: agree at nc={len(CANONICAL_NAMES)}")
else:
    print("NOTE: dataset/final/data.yaml not found -- skipping schema cross-check.")

if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
    print(f"Dropped existing '{DATASET_NAME}' (re-derived from disk, no review work lost).")

dataset = fo.Dataset(DATASET_NAME)
# DEC-081: without this the App blocks on an "Import your dataset schema" prompt.
dataset.classes["ground_truth"] = list(CANONICAL_NAMES)

samples = []
skipped_missing_label = 0
skipped_by_filter = 0
unknown_class_ids: Counter = Counter()

for split in SPLITS:
    images_dir = final_dir(split) / "images"
    labels_dir = final_dir(split) / "labels"
    if not images_dir.is_dir():
        print(f"  {split}: {images_dir} not found -- skipped.")
        continue

    image_paths = sorted(list_images(images_dir, recursive=False))
    if MAX_IMAGES_PER_SPLIT is not None:
        image_paths = image_paths[:MAX_IMAGES_PER_SPLIT]

    kept = 0
    for image_path in image_paths:
        source = _source_of(image_path.name)
        if SOURCE_FILTER is not None and source not in SOURCE_FILTER:
            skipped_by_filter += 1
            continue

        label_path = labels_dir / f"{image_path.stem}.txt"
        if not label_path.is_file():
            # Tracked, not silently dropped -- same posture as merge.py/split.py.
            skipped_missing_label += 1
            continue

        detections = []
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            cx, cy, w, h = (float(v) for v in parts[1:])
            if not (0 <= class_id < len(CANONICAL_NAMES)):
                unknown_class_ids[class_id] += 1
                continue
            # YOLO centre-relative -> FiftyOne top-left-relative.
            detections.append(
                fo.Detection(
                    label=CANONICAL_NAMES[class_id],
                    bounding_box=[cx - w / 2, cy - h / 2, w, h],
                )
            )

        if CLASS_FILTER is not None and not any(d.label in CLASS_FILTER for d in detections):
            skipped_by_filter += 1
            continue

        sample = fo.Sample(filepath=str(image_path))
        sample["source"] = source
        sample["split"] = split
        sample["num_boxes"] = len(detections)
        sample["ground_truth"] = fo.Detections(detections=detections)
        samples.append(sample)
        kept += 1

    print(f"  {split}: {kept} samples")

dataset.add_samples(samples)
dataset.persistent = False  # a viewer, not review state -- regenerated in ~1 min

# Index the fields the App sidebar filters on. NOT an optimisation -- a
# correctness fix, and the reason this cell must re-run it after every rebuild.
#
# FiftyOne 1.20's sidebar searches UNINDEXED fields with a bounded scan and, when
# it runs out of budget, returns a PARTIAL value list with a small "Incomplete
# search. create an index" note under the dropdown. It does not fail or blank out,
# it just quietly shows fewer values than exist -- so the sidebar looked like the
# dataset was missing classes and sources when nothing was missing at all.
#
# Measured here on 2026-09-05, before indexing: `source` offered 5 of its 12 real
# values (every roboflow_* source invisible) and `ground_truth.label` offered 12 of
# 13 (Doors missing). What gets dropped is NOT predictable and differs by field --
# `source` kept the 5 alphabetically-first values, while `ground_truth.label` dropped
# Doors from the MIDDLE of the alphabet (it is the rarest class, 2,280 boxes). So a
# truncated list can look like a complete one; the only reliable tell is the
# "Incomplete search" note. Both lists are complete once indexed.
for _field in ("source", "split", "ground_truth.detections.label"):
    dataset.create_index(_field)
print("Sidebar indexes created (source, split, ground_truth.detections.label) -- "
      "without these the App's filter dropdowns silently show a PARTIAL value list.")

print(f"\nLoaded {len(dataset)} samples across {len(SPLITS)} split(s).")
if skipped_missing_label:
    print(f"WARNING: {skipped_missing_label} image(s) had no matching label file.")
if skipped_by_filter:
    print(f"{skipped_by_filter} image(s) excluded by SOURCE_FILTER/CLASS_FILTER.")
if unknown_class_ids:
    print(f"WARNING: out-of-range class ids found (schema drift?): {dict(unknown_class_ids)}")


RuntimeError: dataset/final/data.yaml disagrees with config/classes.yaml.
  data.yaml    : ['Person', 'Vehicle', 'Motorcycle', 'Pole', 'Animals', 'Shelf', 'Doors', 'Chairs', 'Tables', 'Tricycle', 'Potholes', 'Trash Bins', 'Bicycle', 'Stairs', 'Bench']
  classes.yaml : ['Person', 'Vehicle', 'Motorcycle', 'Pole', 'Animals', 'Shelf', 'Doors', 'Chairs', 'Tables', 'Tricycle', 'Potholes', 'Trash Bins', 'Bicycle']
Re-run scripts/build/generate_yaml.py before browsing.

In [11]:
# ---- Cross-check against split_report.json ----
#
# Confirms this notebook is looking at the same pool split.py actually wrote,
# rather than a stale or partially-copied dataset/final/. Only meaningful on a
# full, unfiltered build.

report_path = reports_dir() / "split_report.json"
if not report_path.is_file():
    print("split_report.json not found -- skipping cross-check.")
elif SOURCE_FILTER or CLASS_FILTER or MAX_IMAGES_PER_SPLIT:
    print("Filters active -- cross-check skipped (counts intentionally differ).")
else:
    report = json.loads(report_path.read_text(encoding="utf-8"))
    expected = report["split_counts"]
    actual = Counter(s["split"] for s in dataset.select_fields("split"))
    ok = True
    for split in SPLITS:
        exp, act = expected.get(split), actual.get(split, 0)
        flag = "OK" if exp == act else "<-- MISMATCH"
        if exp != act:
            ok = False
        print(f"  {split:<6} report={exp:<7} loaded={act:<7} {flag}")
    leakage = report.get("cross_split_duplicate_leakage")
    print(f"\ncross_split_duplicate_leakage: {leakage if leakage else '[] (none)'}")
    print("Counts match split_report.json." if ok else
          "MISMATCH -- dataset/final/ is not what split.py last wrote. Re-run the cascade.")


  train  report=30123   loaded=25606   <-- MISMATCH
  val    report=6645    loaded=5701    <-- MISMATCH
  test   report=6403    loaded=5570    <-- MISMATCH

cross_split_duplicate_leakage: [] (none)
MISMATCH -- dataset/final/ is not what split.py last wrote. Re-run the cascade.


In [12]:
# ---- Per-source x per-split breakdown ----
#
# The table the sidebar cannot give you at a glance. Sorted by total images.

by_source_split: dict[str, Counter] = defaultdict(Counter)
by_source_class: dict[str, Counter] = defaultdict(Counter)
by_class_split: dict[str, Counter] = defaultdict(Counter)

for sample in dataset.select_fields(["source", "split", "ground_truth"]):
    src, split = sample["source"], sample["split"]
    by_source_split[src][split] += 1
    by_source_split[src]["total"] += 1
    seen = {d.label for d in (sample.ground_truth.detections if sample.ground_truth else [])}
    for label in seen:
        by_source_class[src][label] += 1
        by_class_split[label][split] += 1

print("IMAGES BY SOURCE x SPLIT")
print(f"{'source':<42} {'train':>7} {'val':>7} {'test':>7} {'total':>8}")
print("-" * 75)
for src, c in sorted(by_source_split.items(), key=lambda kv: -kv[1]["total"]):
    print(f"{src:<42} {c['train']:>7} {c['val']:>7} {c['test']:>7} {c['total']:>8}")
print("-" * 75)
tot = Counter()
for c in by_source_split.values():
    tot.update(c)
print(f"{'TOTAL':<42} {tot['train']:>7} {tot['val']:>7} {tot['test']:>7} {tot['total']:>8}")

print("\n\nIMAGES BY CLASS x SPLIT   (an image counts once per class it contains)")
print(f"{'class':<14} {'train':>7} {'val':>7} {'test':>7} {'total':>8}   {'val%':>6} {'test%':>6}")
print("-" * 75)
for name in CANONICAL_NAMES:
    c = by_class_split.get(name, Counter())
    t = c["train"] + c["val"] + c["test"]
    vp = f"{c['val']/t:.1%}" if t else "-"
    tp = f"{c['test']/t:.1%}" if t else "-"
    print(f"{name:<14} {c['train']:>7} {c['val']:>7} {c['test']:>7} {t:>8}   {vp:>6} {tp:>6}")


IMAGES BY SOURCE x SPLIT
source                                       train     val    test    total
---------------------------------------------------------------------------
open_images                                  12478    2663    2701    17842
roboflow_dlsu_d_vehicle_type_detection        4332     899     914     6145
exdark                                        3886     907     843     5636
dataset_ninja_road_damage_detector             928     205     197     1330
roboflow_revised_pedestrian_obstacle           926     198     199     1323
roboflow_cv_project_hovyc                      877     191     193     1261
roboflow_pothole_voxrl                         379     149     137      665
dataset_ninja_pothole_detection                405     164      95      664
roboflow_door_detection_zqt59                  449      96      97      642
roboflow_trashcan_detection_pihfn              376     109      74      559
roboflow_roitrikee                             305      65     

In [13]:
# ---- Which classes does each source actually contribute? ----
#
# Worth a look because several sources are no longer single-class: review passes
# added correct boxes for other classes visible in the same photo (DEC-087), so a
# "pothole source" may legitimately carry Person and Vehicle boxes too.

print("CLASSES PER SOURCE (image counts)")
for src, c in sorted(by_source_class.items(), key=lambda kv: -sum(kv[1].values())):
    parts = ", ".join(f"{k}={v}" for k, v in c.most_common())
    print(f"\n  {src}")
    print(f"    {parts}")


CLASSES PER SOURCE (image counts)

  open_images
    Tables=3648, Animals=2918, Bicycle=2887, Shelf=2386, Pole=2154, Chairs=2130, Motorcycle=1636, Trash Bins=1104, Person=1056, Vehicle=539

  roboflow_dlsu_d_vehicle_type_detection
    Vehicle=4184, Motorcycle=2310, Person=1657, Tricycle=1546, Bicycle=117, Chairs=36, Animals=15, Tables=3, Pole=2

  exdark
    Person=2658, Animals=1609, Chairs=1230, Tables=995, Vehicle=906, Bicycle=751, Motorcycle=587

  dataset_ninja_road_damage_detector
    Potholes=1330, Motorcycle=1135, Person=1125, Vehicle=977, Bicycle=41, Chairs=19, Pole=13, Animals=3, Tricycle=2, Tables=2, Doors=1

  roboflow_revised_pedestrian_obstacle
    Person=1218, Vehicle=646, Motorcycle=470, Tricycle=285, Pole=221, Chairs=115, Bicycle=78, Doors=36, Tables=30, Animals=20, Trash Bins=18, Shelf=5

  roboflow_cv_project_hovyc
    Doors=1256, Person=99, Chairs=38, Vehicle=37, Pole=32, Trash Bins=20, Bicycle=17, Tables=11, Animals=5, Motorcycle=3

  roboflow_roitrikee
    Tricycl

## Filtering in the App

Once the App is open, both fields you asked for are in the **left sidebar**:

- **`source`** — every source key (`open_images`, `crowdhuman`, `roboflow_pothole_voxrl`, …).
  Click one to see only that source's images. Multi-select works.
- **`split`** — `train` / `val` / `test`.
- **`ground_truth.label`** — filter by class.
- **`num_boxes`** — a range slider; useful for finding empty or absurdly crowded images.

These compose, so "`source = roboflow_pothole_voxrl` AND `split = val`" is two clicks.

For anything the sidebar can't express, use a view in code — e.g.:

```python
from fiftyone import ViewField as F

view = dataset.match(F("source") == "crowdhuman").match(F("num_boxes") > 20)
session.view = view
```


In [14]:
# Launch the App. If a server is already running (it survives kernel restarts),
# this attaches to it rather than starting a second one.
session = fo.launch_app(dataset, auto=False)
session


Session launched. Run `session.show()` to open the App in a cell output.


Dataset:          final_dataset_browse
Media type:       image
Num samples:      36877
Selected samples: 0
Selected labels:  0
Session URL:      http://localhost:5151/

In [15]:
# Run when finished. The dataset is non-persistent, so it also disappears on
# kernel shutdown -- rebuild from the cells above whenever you want it back.
fo.close_app()
print("App closed.")


App closed.
